# ARC-05: Simulation Diagnostics

Diagnostics for the Newtonian two-body model report numerical drift rather than exact conservation.

- Specific energy: m^2/s^2
- Specific angular momentum: m^2/s
- Eccentricity: dimensionless
- Radius: m
- Speed: m/s


In [1]:
import math
import numpy as np


def _validated_state(state):
    if isinstance(state, (str, bytes, bytearray, bool, np.bool_)):
        raise ValueError("State must be numeric")
    try:
        components = list(state)
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError("State must be numeric") from exc
    for component in components:
        if isinstance(component, (bool, np.bool_, str, bytes, bytearray, complex, np.complexfloating, list, tuple, np.ndarray)):
            raise ValueError("State must contain only real numeric values")
    y = np.asarray(state, dtype=np.float64)
    if y.shape != (6,) or not np.all(np.isfinite(y)):
        raise ValueError("State must be a finite six-component vector")
    return y


def _validated_mu(mu):
    if isinstance(mu, (bool, np.bool_, str, bytes, bytearray)):
        raise ValueError("Gravitational parameter must be numeric")
    mu = float(mu)
    if not math.isfinite(mu) or mu <= 0:
        raise ValueError("Gravitational parameter must be finite and positive")
    return mu


def specific_orbital_energy(state, mu):
    y = _validated_state(state)
    mu = _validated_mu(mu)
    r = float(np.linalg.norm(y[:3]))
    v = float(np.linalg.norm(y[3:]))
    if r <= 0 or not math.isfinite(r) or not math.isfinite(v):
        raise ValueError("State produces an invalid orbital radius or speed")
    return 0.5 * v*v - mu/r


def specific_angular_momentum(state):
    y = _validated_state(state)
    h = np.cross(y[:3], y[3:]).astype(np.float64)
    if not np.all(np.isfinite(h)):
        raise ValueError("Angular momentum contains non-finite values")
    return h, float(np.linalg.norm(h))


def eccentricity_vector(state, mu):
    y = _validated_state(state)
    mu = _validated_mu(mu)
    r_vector = y[:3]
    v_vector = y[3:]
    r = float(np.linalg.norm(r_vector))
    if r <= 0:
        raise ValueError("Position radius must be positive")
    h, _ = specific_angular_momentum(y)
    e_vector = np.cross(v_vector, h)/mu - r_vector/r
    if not np.all(np.isfinite(e_vector)):
        raise ValueError("Eccentricity vector contains non-finite values")
    return e_vector.astype(np.float64), float(np.linalg.norm(e_vector))


def diagnose_trajectory(times_s, states, mu):
    times = np.asarray(times_s, dtype=np.float64)
    trajectory = np.asarray(states, dtype=np.float64)
    mu = _validated_mu(mu)
    if times.ndim != 1 or trajectory.shape != (times.size, 6):
        raise ValueError("Trajectory must contain matching six-component states and times")
    if not np.all(np.isfinite(times)) or not np.all(np.isfinite(trajectory)):
        raise ValueError("Trajectory contains non-finite values")
    if times.size == 0 or np.any(np.diff(times) < 0.0):
        raise ValueError("Times must be non-empty and monotonically ordered")
    energies = np.asarray([specific_orbital_energy(state, mu) for state in trajectory], dtype=np.float64)
    angular_momentum = np.asarray([specific_angular_momentum(state)[1] for state in trajectory], dtype=np.float64)
    eccentricities = np.asarray([eccentricity_vector(state, mu)[1] for state in trajectory], dtype=np.float64)
    radii = np.linalg.norm(trajectory[:, :3], axis=1).astype(np.float64)
    speeds = np.linalg.norm(trajectory[:, 3:], axis=1).astype(np.float64)
    energy_drift = energies - energies[0]
    angular_drift = angular_momentum - angular_momentum[0]
    return {
        "specific_energy_m2_s2": energies,
        "specific_angular_momentum_m2_s": angular_momentum,
        "eccentricity_magnitude": eccentricities,
        "radius_m": radii,
        "speed_m_s": speeds,
        "energy_drift_m2_s2": energy_drift,
        "angular_momentum_drift_m2_s": angular_drift,
        "max_abs_energy_drift_m2_s2": float(np.max(np.abs(energy_drift))),
        "max_abs_angular_momentum_drift_m2_s": float(np.max(np.abs(angular_drift))),
    }


In [2]:
import math
import numpy as np

mu = 3.986004418e14
radius = 7_000_000.0
speed = math.sqrt(mu/radius)
circular = np.array([radius, 0.0, 0.0, 0.0, speed, 0.0], dtype=np.float64)

energy = specific_orbital_energy(circular, mu)
h_vector, h_magnitude = specific_angular_momentum(circular)
e_vector, e_magnitude = eccentricity_vector(circular, mu)
assert np.isfinite(energy)
assert np.isclose(h_vector[2], radius*speed, rtol=1e-12, atol=1e-8)
assert np.isclose(h_magnitude, radius*speed, rtol=1e-12, atol=1e-8)
assert e_magnitude < 1e-10

times = np.array([0.0, 1.0, 2.0], dtype=np.float64)
states = np.vstack((circular, circular, circular)).astype(np.float64)
diagnostics = diagnose_trajectory(times, states, mu)
assert np.allclose(diagnostics["energy_drift_m2_s2"], 0.0)
assert np.allclose(diagnostics["angular_momentum_drift_m2_s"], 0.0)
assert np.allclose(diagnostics["radius_m"], radius)
assert np.allclose(diagnostics["speed_m_s"], speed)

for invalid_state in ([1, 2, 3], [0, 0, 0, 0, 0, 0], [1, 2, 3, 4, 5, "6"]):
    try:
        specific_orbital_energy(invalid_state, mu)
        raise AssertionError("Invalid diagnostic state accepted")
    except ValueError:
        pass

print("Specific energy:", energy, "m^2/s^2")
print("Specific angular momentum magnitude:", h_magnitude, "m^2/s")
print("Eccentricity magnitude:", e_magnitude)
print("Diagnostics validation: PASSED")


Specific energy: -28471460.128571425 m^2/s^2
Specific angular momentum magnitude: 52822373030.75279 m^2/s
Eccentricity magnitude: 0.0
Diagnostics validation: PASSED
